In [1]:
import pickle

def load_pickle(filename):
  with open(filename, 'rb') as f_in:
    return pickle.load(f_in)

X_train, y_train = load_pickle('output/train.pkl')
X_val, y_val = load_pickle('output/val.pkl')
X_test, y_test = load_pickle('output/test.pkl')
dv = load_pickle('output/dv.pkl')

print(X_train.shape, X_val.shape, X_test.shape)

(65946, 5702) (62574, 5702) (69392, 5702)


In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

rf = RandomForestRegressor(
  max_depth=10,
  random_state=0,
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
print(f'RMSE: {rmse:.2f}')

RMSE: 5.43


In [4]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("random_forest_tran")
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="Randomforest_Homework"):
  rf = RandomForestRegressor(max_depth=10, random_state=0)
  rf.fit(X_train, y_train)

  y_pred = rf.predict(X_val)
  rmse = root_mean_squared_error(y_val, y_pred)
  print(rmse)

2026/11/18 12:01:47 INFO mlflow.tracking.fluent: Experiment with name 'random_forest_tran' does not exist. Creating a new experiment.
2026/11/18 12:02:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


5.431151718191326


In [5]:
mlflow.set_experiment("random-forest-hyperopt")

2026/11/18 12:16:08 INFO mlflow.tracking.fluent: Experiment with name 'random-forest-hyperopt' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/MSI/Desktop/MLOps/MLOps_ZoomCamp_Course/02_Experiment_Tracking/mlruns/3', creation_time=1794984369015, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1794984369015, lifecycle_stage='active', name='random-forest-hyperopt', tags={}, trace_location=None, workspace='default'>

In [6]:
mlflow.sklearn.autolog()

In [9]:
import numpy as np
import subprocess
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

def git_info():
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    dirty = subprocess.check_output(["git", "status", "--porcelain"], text=True).strip() != ""
    return commit, str(dirty)

def objective(params):
  with mlflow.start_run():
    commit, dirty = git_info()
    mlflow.set_tag("model", "random_forest")
    mlflow.set_tag("git_commit", commit)
    mlflow.set_tag("git_dirty", dirty)
    mlflow.log_params(params)

    rf = RandomForestRegressor(**params)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

  return {"loss": rmse, "status": STATUS_OK}


In [8]:
search_space = {
        'max_depth': scope.int(hp.quniform('max_depth', 1, 20, 1)),
        'n_estimators': scope.int(hp.quniform('n_estimators', 10, 50, 1)),
        'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 10, 1)),
        'min_samples_leaf': scope.int(hp.quniform('min_samples_leaf', 1, 4, 1)),
        'random_state': 42
    }

In [10]:
best = fmin(
  fn=objective,
  space=search_space,
  algo=tpe.suggest,
  max_evals=15,
  trials=Trials(),
  rstate=np.random.default_rng(42),
)
print(best)


  0%|          | 0/15 [00:00<?, ?trial/s, best loss=?]

2026/11/18 12:22:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  7%|▋         | 1/15 [00:29<06:48, 29.19s/trial, best loss: 5.370086069268862]

2026/11/18 12:23:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



 13%|█▎        | 2/15 [00:49<05:14, 24.21s/trial, best loss: 5.370086069268862]

2026/11/18 12:23:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



... (21 outputs left out)

2026/11/18 12:27:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



 93%|█████████▎| 14/15 [05:40<00:22, 22.93s/trial, best loss: 5.335419588556921]

2026/11/18 12:28:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



100%|██████████| 15/15 [06:11<00:00, 24.79s/trial, best loss: 5.335419588556921]
{'max_depth': np.float64(19.0), 'min_samples_leaf': np.float64(2.0), 'min_samples_split': np.float64(2.0), 'n_estimators': np.float64(11.0)}


In [12]:
import os
import pickle
import click
import mlflow

from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

HPO_EXPERIMENT_NAME = "random-forest-hyperopt"
EXPERIMENT_NAME = "random-forest-best-models"
RF_PARAMS = ['max_depth', 'n_estimators', 'min_samples_split', 'min_samples_leaf', 'random_state']

mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.sklearn.autolog()


def load_pickle(filename):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


def train_and_log_model(data_path, params):

    with mlflow.start_run():
        new_params = {}
        for param in RF_PARAMS:
            new_params[param] = int(params[param])

        rf = RandomForestRegressor(**new_params)
        rf.fit(X_train, y_train)

        # Evaluate model on the validation and test sets
        val_rmse = root_mean_squared_error(y_val, rf.predict(X_val))
        mlflow.log_metric("val_rmse", val_rmse)
        test_rmse = root_mean_squared_error(y_test, rf.predict(X_test))
        mlflow.log_metric("test_rmse", test_rmse)


@click.command()
@click.option(
    "--data_path",
    default="./output",
    help="Location where the processed NYC taxi trip data was saved"
)
@click.option(
    "--top_n",
    default=5,
    type=int,
    help="Number of top models that need to be evaluated to decide which one to promote"
)
def run_register_model(data_path: str, top_n: int):

    client = MlflowClient()

    # Retrieve the top_n model runs and log the models
    experiment = client.get_experiment_by_name(HPO_EXPERIMENT_NAME)
    runs = client.search_runs(
        experiment_ids=experiment.experiment_id,
        run_view_type=ViewType.ACTIVE_ONLY,
        max_results=top_n,
        order_by=["metrics.rmse ASC"]
    )
    for run in runs:
        train_and_log_model(data_path=data_path, params=run.data.params)

    # Select the model with the lowest test RMSE
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
    best_run = client.search_runs(
    experiment_ids=experiment.experiment_id,
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=1,
    order_by=["metrics.test_rmse ASC"],
    )[0]


    # Register the best model
    mlflow.register_model(
    model_uri=f"runs:/{best_run.info.run_id}/model",
    name="random-forest-best-model",
    )



run_register_model.callback("./output", 5)

2026/11/18 12:38:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/11/18 12:38:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/11/18 12:39:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec